
#Tarefa: Gerenciador de Pedidos de Clientes com SQLite

# TAREFA 7
Desenvolvimento Rápido de Aplicações em Python

> Marcos Alcino Ribeiro Cussioli

> 202402394612


In [ ]:
import sqlite3

# ==============================
# Configuração inicial do banco
# ==============================
conexao = sqlite3.connect("empresa.db")
cursor = conexao.cursor()

# Criação das tabelas, se não existirem
cursor.execute("""
CREATE TABLE IF NOT EXISTS clientes (
    id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS pedidos (
    id_pedido INTEGER PRIMARY KEY AUTOINCREMENT,
    id_cliente INTEGER NOT NULL,
    produto TEXT NOT NULL,
    valor REAL NOT NULL,
    FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente)
);
""")
conexao.commit()

# ==============================
# Funções
# ==============================
def cadastrar_cliente():
    nome = input("Digite o nome do cliente: ").strip()
    if not nome:
        print("Nome inválido.")
        return
    cursor.execute("INSERT INTO clientes (nome) VALUES (?)", (nome,))
    conexao.commit()
    print("✅ Cliente cadastrado com sucesso!")

def novo_pedido():
    try:
        id_cliente = int(input("Digite o ID do cliente: "))
        cursor.execute("SELECT * FROM clientes WHERE id_cliente = ?", (id_cliente,))
        cliente = cursor.fetchone()
        if not cliente:
            print("❌ Cliente não encontrado.")
            return
        produto = input("Digite o nome do produto: ").strip()
        valor = float(input("Digite o valor do pedido: "))
        cursor.execute("INSERT INTO pedidos (id_cliente, produto, valor) VALUES (?, ?, ?)",
                       (id_cliente, produto, valor))
        conexao.commit()
        print("✅ Pedido registrado com sucesso!")
    except ValueError:
        print("⚠️ Valor inválido! Digite números corretamente.")

def listar_pedidos_cliente():
    try:
        id_cliente = int(input("Digite o ID do cliente: "))
        cursor.execute("SELECT nome FROM clientes WHERE id_cliente = ?", (id_cliente,))
        cliente = cursor.fetchone()
        if not cliente:
            print("❌ Cliente não encontrado.")
            return
        print(f"\n📋 Pedidos do cliente {cliente[0]}:\n")
        cursor.execute("SELECT id_pedido, produto, valor FROM pedidos WHERE id_cliente = ?", (id_cliente,))
        pedidos = cursor.fetchall()
        if pedidos:
            for p in pedidos:
                print(f"ID Pedido: {p[0]} | Produto: {p[1]} | Valor: R$ {p[2]:.2f}")
        else:
            print("Nenhum pedido encontrado.")
    except ValueError:
        print("⚠️ ID inválido.")

def total_pedidos_cliente():
    try:
        id_cliente = int(input("Digite o ID do cliente: "))
        cursor.execute("SELECT nome FROM clientes WHERE id_cliente = ?", (id_cliente,))
        cliente = cursor.fetchone()
        if not cliente:
            print("❌ Cliente não encontrado.")
            return
        cursor.execute("SELECT SUM(valor) FROM pedidos WHERE id_cliente = ?", (id_cliente,))
        total = cursor.fetchone()[0]
        if total:
            print(f"💰 Valor total dos pedidos de {cliente[0]}: R$ {total:.2f}")
        else:
            print(f"ℹ️ O cliente {cliente[0]} não possui pedidos.")
    except ValueError:
        print("⚠️ ID inválido.")

def listar_clientes():
    cursor.execute("SELECT id_cliente, nome FROM clientes")
    clientes = cursor.fetchall()
    print("\n=== Clientes cadastrados ===")
    if clientes:
        for c in clientes:
            print(f"ID: {c[0]} | Nome: {c[1]}")
    else:
        print("Nenhum cliente cadastrado.")

def listar_todos_pedidos():
    cursor.execute("""
        SELECT c.nome, p.id_pedido, p.produto, p.valor
        FROM pedidos p
        JOIN clientes c ON p.id_cliente = c.id_cliente
        ORDER BY c.nome;
    """)
    pedidos = cursor.fetchall()
    print("\n=== Todos os pedidos ===")
    if pedidos:
        for p in pedidos:
            print(f"Cliente: {p[0]} | ID Pedido: {p[1]} | Produto: {p[2]} | Valor: R$ {p[3]:.2f}")
    else:
        print("Nenhum pedido registrado.")

def total_pedidos_geral():
    cursor.execute("SELECT SUM(valor) FROM pedidos")
    total = cursor.fetchone()[0]
    print("\n=== Valor total de todos os pedidos ===")
    if total:
        print(f"💰 R$ {total:.2f}")
    else:
        print("Nenhum pedido registrado.")

# ==============================
# Menu principal
# ==============================
while True:
    print("\n=== Gerenciador de Pedidos ===")
    print("1. Cadastrar Novo Cliente")
    print("2. Fazer Novo Pedido")
    print("3. Listar Pedidos de um Cliente")
    print("4. Calcular Valor Total de Pedidos de um Cliente")
    print("5. Listar Clientes")
    print("6. Listar Todos os Pedidos")
    print("7. Calcular Total de Todos os Pedidos")
    print("8. Sair")

    opcao = input("Escolha uma opção: ").strip()

    if opcao == "1":
        cadastrar_cliente()
    elif opcao == "2":
        novo_pedido()
    elif opcao == "3":
        listar_pedidos_cliente()
    elif opcao == "4":
        total_pedidos_cliente()
    elif opcao == "5":
        listar_clientes()
    elif opcao == "6":
        listar_todos_pedidos()
    elif opcao == "7":
        total_pedidos_geral()
    elif opcao == "8":
        print("👋 Saindo...")
        break
    else:
        print("⚠️ Opção inválida!")

# Fecha a conexão ao sair
conexao.close()
